In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import plotly.express as px
import plotly.io as pio

from pathlib import Path

In [2]:
DATA_PATH = Path("../data/road_traffic_accidents_zw.csv")

acc_data = pd.read_csv(DATA_PATH)

In [3]:
acc_data["Lat"] = pd.to_numeric(acc_data["Lat"], errors="coerce")
acc_data["Long"] = pd.to_numeric(acc_data["Long"], errors="coerce")

In [4]:
geo_data = acc_data.dropna(subset=["Lat", "Long"]).copy()

In [5]:
coords = np.radians(
    geo_data[["Lat", "Long"]].to_numpy()
)

In [6]:
earth_radius_km = 6371.0088

We shall assume that accidents within a radius of 6 km are 'neighbours'

In [16]:
clust_model = DBSCAN(
    eps=6 / earth_radius_km,
    min_samples=3,
    metric="haversine"
)

In [17]:
geo_data["Cluster"] = clust_model.fit_predict(coords)

print(geo_data["Cluster"].value_counts())

Cluster
-1    149
 0     30
 2     12
 3      9
 1      4
 4      4
 5      3
 6      3
Name: count, dtype: int64


In [18]:
geo_data["Cluster label"] = geo_data["Cluster"].apply(
    lambda x: f"Cluster {x}" if x >= 0 else "Noise"
)

In [19]:

fig = px.scatter_map(
    geo_data,
    lat="Lat",
    lon="Long",
    color="Cluster label",
    hover_name="Location",
    hover_data=["Date", "Road", "Fatalities", "Injured"],
    map_style="open-street-map",
    zoom=5,
    center={"lat": -19.0, "lon": 29.5}
)

fig.show(renderer="browser")

Testing the src files


In [20]:
from pathlib import Path

print("Current working directory:", Path.cwd())

Current working directory: C:\Users\chich\Documents\practice\ML\zw-road-safety-analytics\notebooks


In [21]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [22]:
print("Current working directory:", Path.cwd())

Current working directory: C:\Users\chich\Documents\practice\ML\zw-road-safety-analytics\notebooks


In [23]:
from src.data_loader import load_data
from src.clustering import cluster_accidents
from src.forecasting import forecast_accidents

print("All modules imported successfully!")

All modules imported successfully!


Testing the data loading

In [24]:
df = load_data(source="local")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (214, 17)


C:\Users\chich\Documents\practice\ML\zw-road-safety-analytics\src\data_loader.py:42: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


,ID,Date,Time,Road,Location,City or Town,Province,Injured,Fatalities,Vehicles Involved,Cause,Source,Link,Platform,Documentation,Lat,Long
0,ZW000001,2025-04-22,16:30:00,Mutare-Masvingo,Devure Resettlement Area,NaN,Masvingo,11.0,3,Toyota Hiace,Veered off road,Z.R.P,https://x.com/PoliceZimbabwe/status/1914950174...,X,NaN,-19.971880,31.965905
1,ZW000002,2025-04-25,14:30:00,Harare-Bulawayo,Chegutu,Chegutu,Mashonaland West,9.0,3,Toyota Hiace,Veered off road,Z.R.P,https://x.https://x.com/PoliceZimbabwe/status/...,X,NaN,-17.993871,30.575507
2,ZW000003,2025-04-26,20:30:00,Munetsi Street,Glen Norah A,Harare,Harare,1.0,0,Unknown,Hit and run,Z.R.P,https://x.com/PoliceZimbabwe/status/1916486229...,X,NaN,-17.870562,31.027520
3,ZW000004,2025-04-28,22:30:00,Bristol,Gweru,Gweru,Midlands,0.0,2,Toyota Aqua,Speed chase,Z.R.P,https://x.com/PoliceZimbabwe/status/1917218711...,X,https://x.com/PoliceZimbabwe/status/1917218711...,-19.463604,29.787773
4,ZW000005,2025-04-28,23:00:00,Robert Mugabe Way,Eastlea,Harare,Harare,0.0,1,Unknown,Hit and run,Z.R.P,https://x.com/PoliceZimbabwe/status/1917251996...,X,NaN,-17.830363,31.063315


Testing the clustering block

In [27]:
clustered_df = cluster_accidents(
    df,
    radius_km=6,
    min_samples=3
)

display(
    clustered_df[
        ["Date", "Province", "Lat", "Long", "Cluster"]
    ].head(10)
)

print(
    "Clusters found:",
    clustered_df.loc[
        clustered_df["Cluster"] >= 0, "Cluster"
    ].nunique()
)

,Date,Province,Lat,Long,Cluster
0,2025-04-22,Masvingo,-19.971880,31.965905,-1
1,2025-04-25,Mashonaland West,-17.993871,30.575507,-1
2,2025-04-26,Harare,-17.870562,31.027520,0
3,2025-04-28,Midlands,-19.463604,29.787773,1
4,2025-04-28,Harare,-17.830363,31.063315,0
5,2025-05-02,Mashonaland East,-18.167131,31.796205,-1
6,2025-05-03,Matabeleland South,-19.925527,29.029839,-1
7,2025-05-03,Manicaland,-18.936488,32.622639,-1
8,2025-05-04,Midlands,-18.795523,29.805714,-1
9,2025-05-07,Midlands,-19.471190,29.803125,1


Clusters found: 7


In [30]:
import plotly.express as px

clustered_df["Cluster label"] = clustered_df["Cluster"].apply(
    lambda x: f"Cluster {x}" if x >= 0 else "Noise"
)

fig = px.scatter_map(
    clustered_df,
    lat="Lat",
    lon="Long",
    color="Cluster label",
    hover_name="Location",
    hover_data=["Date", "Fatalities", "Injured"],
    map_style="open-street-map",
    zoom=5,
    center={"lat": -19.0, "lon": 29.5},
    height=600
)

fig.show(renderer = "browser")

Testing the forecasting block

In [31]:
from src.data_loader import load_data
from src.forecasting import forecast_accidents

df = load_data(source="local")

forecast_df = forecast_accidents(df, horizon=8)

display(forecast_df.tail(12))

C:\Users\chich\Documents\practice\ML\zw-road-safety-analytics\src\data_loader.py:42: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


,Week,Accidents,Type
71,2026-09-06,1.000000,Historical
72,2026-09-13,5.000000,Historical
73,2026-09-20,1.000000,Historical
74,2026-09-27,2.000000,Historical
75,2026-10-04,2.365702,Forecast
76,2026-10-11,2.365702,Forecast
77,2026-10-18,2.365702,Forecast
78,2026-10-25,2.365702,Forecast
79,2026-11-01,2.365702,Forecast
80,2026-11-08,2.365702,Forecast


In [33]:
import plotly.express as px

fig = px.line(
    forecast_df,
    x="Week",
    y="Accidents",
    color="Type",
    markers=True,
    title="Weekly Road Traffic Accident Forecast"
)

fig.show(renderer = "browser")